--- Day 12: Garden Groups ---
Why not search for the Chief Historian near the gardener and his massive farm? There's plenty of food, so The Historians grab something to eat while they search.

You're about to settle near a complex arrangement of garden plots when some Elves ask if you can lend a hand. They'd like to set up fences around each region of garden plots, but they can't figure out how much fence they need to order or how much it will cost. They hand you a map (your puzzle input) of the garden plots.

Each garden plot grows only a single type of plant and is indicated by a single letter on your map. When multiple garden plots are growing the same type of plant and are touching (horizontally or vertically), they form a region. For example:

AAAA
BBCD
BBCC
EEEC
This 4x4 arrangement includes garden plots growing five different types of plants (labeled A, B, C, D, and E), each grouped into their own region.

In order to accurately calculate the cost of the fence around a single region, you need to know that region's area and perimeter.

The area of a region is simply the number of garden plots the region contains. The above map's type A, B, and C plants are each in a region of area 4. The type E plants are in a region of area 3; the type D plants are in a region of area 1.

Each garden plot is a square and so has four sides. The perimeter of a region is the number of sides of garden plots in the region that do not touch another garden plot in the same region. The type A and C plants are each in a region with perimeter 10. The type B and E plants are each in a region with perimeter 8. The lone D plot forms its own region with perimeter 4.

Visually indicating the sides of plots in each region that contribute to the perimeter using - and |, the above map's regions' perimeters are measured as follows:

+-+-+-+-+
|A A A A|
+-+-+-+-+     +-+
              |D|
+-+-+   +-+   +-+
|B B|   |C|
+   +   + +-+
|B B|   |C C|
+-+-+   +-+ +
          |C|
+-+-+-+   +-+
|E E E|
+-+-+-+
Plants of the same type can appear in multiple separate regions, and regions can even appear within other regions. For example:

OOOOO
OXOXO
OOOOO
OXOXO
OOOOO
The above map contains five regions, one containing all of the O garden plots, and the other four each containing a single X plot.

The four X regions each have area 1 and perimeter 4. The region containing 21 type O plants is more complicated; in addition to its outer edge contributing a perimeter of 20, its boundary with each X region contributes an additional 4 to its perimeter, for a total perimeter of 36.

Due to "modern" business practices, the price of fence required for a region is found by multiplying that region's area by its perimeter. The total price of fencing all regions on a map is found by adding together the price of fence for every region on the map.

In the first example, region A has price 4 * 10 = 40, region B has price 4 * 8 = 32, region C has price 4 * 10 = 40, region D has price 1 * 4 = 4, and region E has price 3 * 8 = 24. So, the total price for the first example is 140.

In the second example, the region with all of the O plants has price 21 * 36 = 756, and each of the four smaller X regions has price 1 * 4 = 4, for a total price of 772 (756 + 4 + 4 + 4 + 4).

Here's a larger example:

RRRRIICCFF
RRRRIICCCF
VVRRRCCFFF
VVRCCCJFFF
VVVVCJJCFE
VVIVCCJJEE
VVIIICJJEE
MIIIIIJJEE
MIIISIJEEE
MMMISSJEEE
It contains:

A region of R plants with price 12 * 18 = 216.
A region of I plants with price 4 * 8 = 32.
A region of C plants with price 14 * 28 = 392.
A region of F plants with price 10 * 18 = 180.
A region of V plants with price 13 * 20 = 260.
A region of J plants with price 11 * 20 = 220.
A region of C plants with price 1 * 4 = 4.
A region of E plants with price 13 * 18 = 234.
A region of I plants with price 14 * 22 = 308.
A region of M plants with price 5 * 12 = 60.
A region of S plants with price 3 * 8 = 24.
So, it has a total price of 1930.

What is the total price of fencing all regions on your map?

In [15]:
"""
I need
- a region builder that assigns each plot to a region and keeps track of that mapping
- a perimeter counter that counts the number of edges a given plot has that are not its own region
"""

'\nI need\n- a region builder that assigns each plot to a region and keeps track of that mapping\n- a perimeter counter that counts the number of edges a given plot has that are not its own region\n'

In [16]:
with open('input12.txt') as file:
    data = [line.rstrip() for line in file]
data[0], len(data), len(data[0])

('EEEEEEEEUUURRRRNRMMOJOOOOSSSSSSSSSSSSSEERRWWOOOOOOOOOOTTTTTCCCCCCRRYYYYYGGGGGGGGGGGGAAAAAAAAAAAAAPPPPMMMUUUUUUUUUURRRRRRRRRRRCCCCCCCHHHHHHHH',
 140,
 140)

In [17]:
"""our goal is a layout like this:
regions = set(region_1, region_2, etc)
plot_dict[plot_coord] = region_x
region_x = set(plot_1, plot_2, plot_3, etc)
plant_dict[region_1] = "A"

I think with this many attributes it might be easier to do this with classes.
For the plot class, its attributes are coordinates, type and region it belongs to

For the region class, attributes are its type and its members
"""

'our goal is a layout like this:\nregions = set(region_1, region_2, etc)\nplot_dict[plot_coord] = region_x\nregion_x = set(plot_1, plot_2, plot_3, etc)\nplant_dict[region_1] = "A"\n\nI think with this many attributes it might be easier to do this with classes.\nFor the plot class, its attributes are coordinates, type and region it belongs to\n\nFor the region class, attributes are its type and its members\n'

In [ ]:
class Plot:

    def __init__(self, coord, plant_type, plot_map):
        self.coord = coord
        self.x, self.y = coord
        self.plant = plant_type
        self.map = plot_map
        self.region = None


    def __repr__(self):
        return f"Plot {self.coord}"

    def find_neighbors(self):
        neighbor_coords = [ (self.x + 1, self.y),
                            (self.x - 1, self.y),
                            (self.x, self.y + 1),
                            (self.x, self.y - 1)]
        
        self.neighbors = set()
        for ne in neighbor_coords:
            if ne in self.map:
                self.neighbors.add(self.map[ne])

    @property
    def perimeter(self):
        NUM_BORDERS = 4
        shared_ne = 0
        for ne in self.neighbors:
            if ne.region == self.region:
                shared_ne += 1 
        return NUM_BORDERS - shared_ne
    
    @property
    def edges(self):
        neighbor_coords = [ (self.x + 1, self.y),
                    (self.x - 1, self.y),
                    (self.x, self.y + 1),
                    (self.x, self.y - 1)]
        
        edges_ = {(self.coord, ne) for ne in neighbor_coords}

        for ne in self.neighbors:
            if ne.region == self.region:
                edges_.remove((self.coord, ne.coord))
        return edges_


class Region:

    def __init__(self, plant_type, child, region_set):
        self.plant = plant_type
        self.regions = region_set
        self.children = {child}

    @staticmethod
    def build_regions(plot_map):
        region_set = set()
        for plot in plot_map.values():
            plot.region = Region(plot.plant, plot, region_set)
            region_set.add(plot.region)


        def merge(from_region, into_region):
            into_region.children.update(from_region.children)
            for plot in from_region.children:
                plot.region = into_region
            region_set.remove(from_region)

        visited = set()
        for plot in plot_map.values(): # iterating through all plots once should suffice
            plot.find_neighbors()
            visited.add(plot)
            for ne in plot.neighbors:
                if all((ne not in visited,
                       plot.plant == ne.plant,
                       plot.region != ne.region)):
                        
                        merge(plot.region, ne.region)
        
        return region_set
    
    @property
    def area(self):
        return len(self.children)
    
    @property
    def perimeter(self):
        return sum([plot.perimeter for plot in self.children])
    
    @property
    def straight_line_perimeter(self):
        edges = set().union(*[plot.edges for plot in self.children])
        if len(edges) != self.perimeter:
            raise ValueError
        line_count = len(edges)
        for (in_x, in_y), (out_x, out_y) in edges:
            if in_x == out_x: # vertical line
                if ((in_x + 1, in_y), (out_x + 1, out_y)) in edges:
                    line_count -= 0.5
                if ((in_x - 1, in_y), (out_x - 1, out_y)) in edges:
                    line_count -= 0.5
            elif in_y == out_y: # horizontal line
                if ((in_x, in_y + 1), (out_x, out_y + 1)) in edges:
                    line_count -= 0.5
                if ((in_x, in_y - 1), (out_x, out_y - 1)) in edges:
                    line_count -= 0.5
        if line_count % 1 != 0:
            raise ValueError
        return int(line_count)




In [19]:
plot_map = {}
for x, line in enumerate(data):
    for y, char in enumerate(line):
        plot_map[(x, y)] = Plot((x, y), char, plot_map)
regions = Region.build_regions(plot_map)

In [20]:
# let's see if we can visualize the regions we identified:
from itertools import cycle
symbol = cycle(range(10))
symbols_for_regions = {r : s for r, s in zip(regions, symbol)}

for x, line in enumerate(data):
    for y, char in enumerate(line):
        print(symbols_for_regions[plot_map[(x, y)].region], end="")
    print()

88888888111999909995455552222222222222337700222222222244444555555117777711111111111100000000000007777999000000000055555555555666666655555555
88888888819999999995555552222223233333333300022222244444444555555777777711111111111100000000000000007999900000000055555555565666666556555555
88888888880099999995555555552233333333333000020220224444444555555777771111111111111100000000000070077999999090000005555556666666666666555555
88888888880099999555555885522333333333336000000000224444444555557767111111111111111000000000000077777919999999000000555555666666666666555555
88888888800099955555555555522333333333330000000000200144455555555761111111111111111000000000000077777711999099000000555555566666666666655555
88888888800089555555555554433333333333000080000000000044444555555551111111111111110000000000000777777777797777711000005888866688866666665555
88888888888889999995555555433363333333000000000000000004425555555555111111111111110007700000000007777777777777771100008888888888866665555555
8888888888889

In [21]:
# looks good to me! let's move on to the next part
total_cost = 0
for r in regions:
    cost = r.area * r.perimeter
    total_cost += cost
total_cost

1483212

Your puzzle answer was 1483212.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
Fortunately, the Elves are trying to order so much fence that they qualify for a bulk discount!

Under the bulk discount, instead of using the perimeter to calculate the price, you need to use the number of sides each region has. Each straight section of fence counts as a side, regardless of how long it is.

Consider this example again:

AAAA
BBCD
BBCC
EEEC
The region containing type A plants has 4 sides, as does each of the regions containing plants of type B, D, and E. However, the more complex region containing the plants of type C has 8 sides!

Using the new method of calculating the per-region price by multiplying the region's area by its number of sides, regions A through E have prices 16, 16, 32, 4, and 12, respectively, for a total price of 80.

The second example above (full of type X and O plants) would have a total price of 436.

Here's a map that includes an E-shaped region full of type E plants:

EEEEE
EXXXX
EEEEE
EXXXX
EEEEE
The E-shaped region has an area of 17 and 12 sides for a price of 204. Including the two regions full of type X plants, this map has a total price of 236.

This map has a total price of 368:

AAAAAA
AAABBA
AAABBA
ABBAAA
ABBAAA
AAAAAA
It includes two regions full of type B plants (each with 4 sides) and a single region full of type A plants (with 4 sides on the outside and 8 more sides on the inside, a total of 12 sides). Be especially careful when counting the fence around regions like the one full of type A plants; in particular, each section of fence has an in-side and an out-side, so the fence does not connect across the middle of the region (where the two B regions touch diagonally). (The Elves would have used the Möbius Fencing Company instead, but their contract terms were too one-sided.)

The larger example from before now has the following updated prices:

A region of R plants with price 12 * 10 = 120.
A region of I plants with price 4 * 4 = 16.
A region of C plants with price 14 * 22 = 308.
A region of F plants with price 10 * 12 = 120.
A region of V plants with price 13 * 10 = 130.
A region of J plants with price 11 * 12 = 132.
A region of C plants with price 1 * 4 = 4.
A region of E plants with price 13 * 8 = 104.
A region of I plants with price 14 * 16 = 224.
A region of M plants with price 5 * 6 = 30.
A region of S plants with price 3 * 6 = 18.
Adding these together produces its new total price of 1206.

What is the new total price of fencing all regions on your map?

In [22]:
'''How do we count "sides" of a region? Well we count the number of straight edges. How do we do that?
I think we start with identifying outside edges precisely with the bordering IN and OUT coordinates (OUT coordinates can be outside the grid). Then we check if the plots one over (that is both IN and OUT extended by 1 or -1) is also a border for that region. If it is, we count the grid as 0.5 (the other one will also count as 0.5). If both sides are part of the same border line, we count 0 (so we subtract 0.5 per straight line segment in either direction)
'''

'How do we count "sides" of a region? Well we count the number of straight edges. How do we do that?\nI think we start with identifying outside edges precisely with the bordering IN and OUT coordinates (OUT coordinates can be outside the grid). Then we check if the plots one over (that is both IN and OUT extended by 1 or -1) is also a border for that region. If it is, we count the grid as 0.5 (the other one will also count as 0.5). If both sides are part of the same border line, we count 0 (so we subtract 0.5 per straight line segment in either direction)\n'

In [23]:
total_cost = 0
for r in regions:
    cost = r.area * r.straight_line_perimeter
    total_cost += cost
total_cost

897062

That's the right answer! You are one gold star closer to finding the Chief Historian.

You have completed Day 12! You can [Share] this victory or [Return to Your Advent Calendar].